# Full Pipeline

End-to-end: data loading → GP prior → instrument response → noise → likelihood → inference → posterior analysis.

## Setup

## Config

In [ ]:
cfg = {
    "gp": {
        "offset_mean":    0.0,
        "offset_std":     (1.0,  0.1),
        "fluctuations":   (1.0,  0.1),
        "loglogavgslope": (-3.0, 1.0),
        "flexibility":    (0.5,  0.1),
    },
    "mixture": {
        "mix_mode": "full",
        "mix_full": {
            "prior_type":   "normal",
            "prior_params": [0.0, 1.0],   # shared across all matrix elements
        },
        "mix_offset": {
            "prior_type":   "normal",
            "prior_params": [0.0, 1.0],
        },
    },
    "params": {
        "background_std": {
            "prior_type":   "lognormal",
            "prior_params": [[0.01, 0.1], [0.01, 0.1], [0.01, 0.1], [0.01, 0.1]],
            "mode":         "sample",
            "freeze_until": 5,
        },
        "poisson_scale": {
            "prior_type":   "lognormal",
            "prior_params": [[1.0, 0.5], [1.0, 0.5], [1.0, 0.5], [1.0, 0.5]],
            "mode":         "sample",
            "freeze_until": 5,
        },
        "psf_sigma": {
            "prior_type":   "lognormal",
            "prior_params": [[1.0, 0.1], [1.0, 0.1], [1.0, 0.1], [1.0, 0.1]],
            "mode":         "point_estimate",
            "freeze_until": 0,
        },
        "psf_rotation": {
            "prior_type":   "normal",
            "prior_params": [[0.0, 0.1], [0.0, 0.1], [0.0, 0.1], [0.0, 0.1]],
            "mode":         "point_estimate",
            "freeze_until": 0,
        },
    },
    "inference": {
        "n_iterations":     10,
        "n_samples":        2,
        "output_directory": "output/run_001",
        "seed":             42,
    },
}

## 1. Data

In [ ]:
from astroprism.io import load_dataset

dataset = load_dataset(
    path="../data/tutorial/jwst_miri_cutout/",
    instrument="JWST_MIRI",
    extension="fits",
)
print(dataset.summary())

In [ ]:
fig, axes = plt.subplots(1, dataset.n_channels, figsize=(4 * dataset.n_channels, 4))
for i, key in enumerate(dataset.channel_keys):
    data, wcs, psf = dataset[key]
    axes[i].imshow(data, norm=LogNorm(), origin='lower')
    axes[i].set_title(key)
plt.suptitle("Data")
plt.tight_layout()
plt.show()

## 2. Model

In [ ]:
from astroprism.models.gp import SpatialGP, MixtureGP
from astroprism.models.response import InstrumentResponse
from astroprism.models.noise import NoiseModel
from astroprism.models.forward import ForwardModel

p = cfg["params"]
m = cfg["mixture"]

spatial_gp = SpatialGP(
    n_channels=dataset.n_channels,
    shape=dataset.shapes[0],
    distances=dataset.pixel_scales[0],
    **cfg["gp"],
)
mixture = MixtureGP(
    spatial_gps=spatial_gp,
    mix_mode=m["mix_mode"],
    mix_full=m["mix_full"],
    mix_offset=m["mix_offset"],
)

response = InstrumentResponse(
    dataset=dataset,
    signal_wcs=dataset.wcs[0],
    signal_shape=dataset.shapes[0],
    psf_sigma=p["psf_sigma"],
    psf_rotation=p["psf_rotation"],
)

noise = NoiseModel(
    n_channels=dataset.n_channels,
    background_std=p["background_std"],
    poisson_scale=p["poisson_scale"],
)

model = ForwardModel(mixture, response, noise)
print("Parameters:", list(model.domain.keys()))

## 3. Likelihood

In [ ]:
from astroprism.models.likelihood import build_likelihood

likelihood = build_likelihood(dataset, model, mask=dataset.readout)

## 4. Inference

`constants` freezes params completely. `point_estimates` MAP-estimates them (optimised but not sampled).
Both are `callable(iteration) -> tuple[str]` — return empty tuple to unfreeze.

In [ ]:
from astroprism.inference import VariationalInference

def build_constants(params):
    """Params that are frozen (constant) at iteration i."""
    return lambda i: tuple(
        k for k, v in params.items()
        if v["mode"] == "constant" or i < v.get("freeze_until", 0)
    )

def build_point_estimates(params):
    """Params that are MAP-estimated (not sampled) at iteration i."""
    return lambda i: tuple(
        k for k, v in params.items()
        if v["mode"] == "point_estimate" and i >= v.get("freeze_until", 0)
    )

vi = VariationalInference(likelihood, seed=cfg["inference"]["seed"])

samples, state = vi.run(
    n_iterations=cfg["inference"]["n_iterations"],
    n_samples=cfg["inference"]["n_samples"],
    output_directory=cfg["inference"]["output_directory"],
    constants=build_constants(cfg["params"]),
    point_estimates=build_point_estimates(cfg["params"]),
)

## 5. Posterior Analysis

If running in a fresh session (inference already done), reload samples from checkpoint instead of re-running:

In [ ]:
# # Reload from checkpoint (use this in a new session instead of re-running inference)
# import pickle
# from pathlib import Path
# samples, state = pickle.load(open(Path("output/run_001") / "last.pkl", "rb"))

In [ ]:
# Posterior mean parameters
posterior_mean = jft.mean(samples)

# Predicted signal (sky before instrument effects)
pred_signal = mixture(posterior_mean)

# Predicted data (signal after PSF + reprojection)
pred_data = response(posterior_mean, pred_signal)

# Residuals
residuals = jnp.array(dataset.data) - jnp.array(pred_data)

In [ ]:
def plot_channels(images, title, norm=None):
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]
    for i, (ax, img) in enumerate(zip(axes, images)):
        ax.imshow(img, origin='lower', norm=norm)
        ax.set_title(dataset.channel_keys[i])
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

plot_channels(dataset.data, "Observed Data", norm=LogNorm())
plot_channels(pred_signal, "Predicted Signal (Sky)", norm=LogNorm())
plot_channels(pred_data, "Predicted Data", norm=LogNorm())
plot_channels(residuals, "Residuals (Observed - Predicted)")

In [ ]:
# Posterior samples — spread shows uncertainty
signal_samples = [mixture(s) for s in samples.samples]
signal_std = jnp.std(jnp.stack(signal_samples), axis=0)

plot_channels(signal_std, "Posterior Std (Signal Uncertainty)", norm=LogNorm())